In [ ]:
# ============================================================
# REGRAS DA LÓGICA
# ============================================================
# ~   : negação
# &   : conjunção
# |   : disjunção
# ->  : implicação/condicional
# <-> : bicondicional
# Toda operação binária deve estar entre parênteses
# ============================================================

import pandas as pd                                         

# Identifica qual o operador principal da expressão, baseado no seu nivél de profundidade dos parênteses, ou seja, o op só pode estar no nivel 0.
def operador_principal(expr): # expr é a string contendo a expressão lógica que será analisada

    nivel = 0 # O nível controla a profundidade dos parênteses, onde 0 significa que está fora deles

    for i in range(len(expr)): # Percorre toda expressão
        if expr[i] == '(': # Indica que entrou em uma subexpressão, logo o op não pode estar aqui. (A&B)->C
            nivel += 1 

        elif expr[i] == ')': # Saiu da subexpressão 
            nivel -= 1 

        elif nivel == 0: # Só analisa os operadores se estiver fora de qualquer parêntese (nível zero)
            # Faz a verificação do operador
            if expr[i:i+3] == '<->': # Faz o fatiamento de 3 caracteres para checar o operador bicondicional
                return '<->', i # Retorna o operador e sua posição encontrada

            if expr[i:i+2] == '->': # Faz o fatiamento de 2 caracteres para checar o operador de implicação
                return '->', i # # Retorna o operador e sua posição encontrada

            if expr[i] == '&':
                return '&', i

            if expr[i] == '|':
                return '|', i 

    return None, None # Se o loop terminar e nenhum operador for achado no nível 0, retorna valores nulos


def avaliar(expr, valores): # expr é a expressão a ser calculada e valores é o dicionário com os valores (True/False) das variáveis
    """
    Avalia recursivamente a expressão lógica com base nos valores booleanos atribuídos às variáveis:
    """
    expr = remover_parenteses_externos(expr) # Remove os parênteses redundantes

    # CASO BASE
    if len(expr) == 1: # (A,B,C) somente a variável
        return valores[expr] # Retorna o seu valor booleano armazenado em valores

    if expr[0] == '~':
        return not avaliar(expr[1:], valores) # Se começar com negação, é aplicado o not e seu valor é invertido

    op, pos = operador_principal(expr) # Aramzena o operador principal e a sua posição

    if op is None:
        raise Exception(f"Operador principal não encontrado em {expr}") # Verifica se o op existe, se não retorna erro

    # DIVISÃO DA EXPRESSÃO
    esquerda = expr[:pos] # Pega a subexpressão a esquerda do operador principal
    direita = expr[pos + len(op):] # Pega a subexpressão a direita do operador principal

    # CHAMADA RECURSIVA
    a = avaliar(esquerda, valores) # Faz uma chamada recursiva até cair no caso base, ou seja, ficar somente uma variável (A) e armezena o seu valor booleano em a
    b = avaliar(direita, valores) 

    # Identificando qual o operador e realizando a operação em python
    if op == '&': # Conjução
        return a and b # Retorna a AND b

    elif op == '|': # Disjunção
        return a or b # Retorna a OU b

    elif op == '->': # Condicional
        return (not a) or b # Retorna a equivalência da condicional, que é a negação do primeiro ou o segundo

    elif op == '<->': # Bicondicional
        return a == b # Retorna True se os dois lados forem iguais (ambos True ou ambos False)


# REMOVER PARÊNTESES REDUNDANTES. EX: ((A&B))
def remover_parenteses_externos(expr):
    while True:
        if len(expr) < 2 or expr[0] != "(" or expr[-1] != ")":
            #Verifica se a expressão começa ou termina com parênteses
            return expr # Se não já retorna a expressão, pois não há parênteses para serem removidos

        nivel = 0 # Nível é um contador de parênteses abertos
        pode_remover = True

        for i in range(len(expr)): # Percorre toda a expressão verificando os parênteses que abriram e fecharam
            if expr[i] == "(": 
                nivel += 1 
            elif expr[i] == ")":
                nivel -= 1 

            if nivel == 0 and i != len(expr) - 1:
                # Se o nível zerou antes do ultimo caracter da expressão, se sim, então os parênteses não envolvem toda expressão e NÂO podem ser removidos.(P&Q)->R
                pode_remover = False
                break

        if pode_remover:
            expr = expr[1:-1] # Faz o slicing para tirar o primeiro e o último dígito, que são os parênteses redundantes

        else:
            return expr
            # Se não retorna a expresão com os parênteses, pois eles são obrigatórios para manter a ordem da expressão

#  4 - MEU
def encontrar_relacoes(expr,coluna_legenda): # Coluna legenda é a lista onde todas as subexpressões encontradasm serão armazenadas
    # Extrai subexpressões entre parênteses:

    pilha = [] # A pilha guarda as posições dos parênteses de abertura

    for i in range(len(expr)): # i é o índice do caractere atual
        if expr[i] == "(":
            pilha.append(i) # Guarda o índice na pilha, e continu aprocurando por outros parênteses do tipo "("

        elif expr[i] == ")": #  Retira o último parêntese da pilha para evitar experessões inválidas, como "((A&B)"
            inicio = pilha.pop() # Inicio é o índice (a posição) do parêntese de abertura que acabou de ser encontrado

            # ignora expressão principal inteira
            if inicio == 0 and i == len(expr) - 1:
                continue

            sub = expr[inicio:i + 1] # Extrai a subexpressão a ser guardada, usando i + 1 para incluir o parêntese de fechamento ')'
            legenda = remover_parenteses_externos(sub) # Rempove os parênyeses externos da subexpressão específica 

            # Evita duplicação de subexpressões 
            if legenda not in coluna_legenda:
                coluna_legenda.append(legenda) # Adiciona à lista "coluna_legenda"
                encontrar_relacoes(sub,coluna_legenda) # Procura subexpressões dentro dela de forma recursiva

# 5 - MEU
def numero_variaveis_linhas(coluna_legenda): # coluna_legenda é a lista contendo todos os componentes da tabela verdade
    # Conta variáveis puras e define tamanho da tabela:

    variaveis = 0 # Inicializa o contador que vai armazenar a quantidade de variáveis proposicionais únicas

    for elemento in coluna_legenda: # elemento representa cada subexpressão ou variável individual da lista
        if len(elemento) == 1 and elemento.isalpha(): # Verifica se o elemento tem tamanho 1 e é uma letra do alfabeto (variável pura)
            variaveis += 1 # Incrementa o contador de variáveis em 1 ao achar uma letra isolada

    return variaveis, 2 ** variaveis # Retorna a quantidade total de variáveis e o número de linhas necessárias (2 elevado a n)

# 6 - MEU
def gerar_combinacoes_bool(n, tabela): # n é a quantidade de variáveis e tabela é a lista onde as linhas de True/False serão salvas
    # Gera todas combinações possíveis de True/False:

    for i in range(2 ** n): # i vai de 0 até o total de linhas menos 1, representando cada linha no formato numérico binário
        linha = [] # Cria uma lista vazia para guardar a combinação de valores booleanos da linha atual

        for j in range(n - 1, -1, -1): # j percorre as posições dos bits de trás para frente para manter a ordem padrão das tabelas verdade
            linha.append(bool((i >> j) & 1)) # Usa deslocamento de bits (>>) e máscara (& 1) para extrair o valor lógico e adicioná-lo à linha

        tabela.append(linha) # Adiciona a linha preenchida com os booleanos dentro da estrutura da matriz "tabela"

    return tabela # Retorna a tabela completa preenchida com a matriz booleana


def gerar_bool_subexpressões(coluna_legenda,tabela): # coluna_legenda possui os nomes das colunas e tabela é a matriz com as combinações lógicas
    # Cria uma lista contendo apenas as variáveis proposicionais unitárias (letras isoladas):

    variaveis = [x for x in coluna_legenda
                 if len(x)==1 and x.isalpha()]

    idx = {} # Dicionário para mapear o nome de cada coluna ao seu respectivo índice numérico

    for i,nome in enumerate(coluna_legenda): # i é o índice numérico e nome é o texto da subexpressão atual
        idx[nome]=i # Associa o nome da coluna ao seu índice para buscas rápidas mais adiante

    for linha in tabela[1:]: # Percorre as linhas da tabela, pulando a primeira linha de cabeçalho

        while len(linha)<len(coluna_legenda): # Enquanto a linha atual for menor que a quantidade total de colunas necessárias
            linha.append(None) # Preenche os espaços das subexpressões com None para que a linha cresça até o tamanho correto

    for i,linha in enumerate(tabela): # i representa o índice da linha atual e linha é a lista de valores dessa linha

        if i==0:
            continue # Pula a primeira linha (índice 0) caso ela corresponda ao cabeçalho de textos da tabela

        valores = {} # Dicionário temporário para armazenar o mapeamento atual de cada variável para seu valor True/False

        for v in variaveis: # v representa cada uma das variáveis puras encontradas no início da função
            valores[v]=linha[idx[v]] # Guarda no dicionário o valor booleano atual que a variável possui nesta linha específica

        for coluna in coluna_legenda: # coluna representa o nome do cabeçalho ou subexpressão que está sendo processada

            if len(coluna)==1:
                continue # Pula se for uma variável simples, pois o valor dela já está preenchido na tabela

            linha[idx[coluna]] = avaliar(coluna,valores) # Calcula o valor lógico da subexpressão e salva o resultado na sua respectiva coluna

    return tabela # Retorna a matriz da tabela verdade completamente preenchida com os resultados de todas as subexpressões

def programa():
    # ============================================================
    # ENTRADA
    # ============================================================
    expressao = input("Digite a expressão lógica: ").replace(" ", "") # Captura a entrada do usuário e remove todos os espaços em branco da string

    # validação de caracteres
    contador_invalidos=0 # Inicializa o contador para identificar se existem símbolos não permitidos na expressão
    for c in expressao: # c representa cada caractere individual da expressão digitada
        caracteres= [
        'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M',
        'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z',
        'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm',
        'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z','(',')','&','|','<','>','-','~'
    ]
        if c not in caracteres:
            contador_invalidos+=1 # Incrementa o contador caso o caractere atual não pertença à lista de símbolos válidos
            break # Interrompe o laço imediatamente ao encontrar o primeiro erro de digitação
    if contador_invalidos==1:
        print('Expressão Inválida (Presença de caracteres indevidos)') # Exibe uma mensagem de alerta se houver caracteres inválidos
    else:
        # validação de parênteses
        pilha = [] # A pilha guarda temporariamente os parênteses de abertura para verificar o fechamento correto
        erro = False # Flag para sinalizar se a estrutura de parênteses está incorreta

        for c in expressao:
            if c == "(":
                pilha.append(c) # Empilha o parêntese de abertura para validação futura
            elif c == ")":
                if not pilha:
                    erro = True # Se encontrar um ")" sem nenhum "(" correspondente na pilha, define erro como verdadeiro
                    break
                pilha.pop() # Remove o parêntese correspondente da pilha ao encontrar seu par de fechamento

        if erro or pilha:
            print("Erro: parênteses inválidos.") # Informa o erro se a flag for True ou se sobrar algum parêntese aberto na pilha

        else:

            # garante expressão bem formada
            if expressao[0] != "(" or expressao[-1] != ")":
                expressao = "(" + expressao + ")" # Envolve toda a expressão em parênteses caso ela já não comece e termine com eles

            expressao = remover_parenteses_externos(expressao) # Limpa os parênteses redundantes das extremidades da string atualizada

            # ========================================================
            # CONSTRUÇÃO DO CABEÇALHO (COLUNAS DA TABELA)
            # ========================================================
            coluna_legenda = [] # Inicializa a lista que guardará todos os títulos (cabeçalhos) das colunas da tabela verdade

            # variáveis puras
            for c in expressao:
                if c.isalpha() and c not in coluna_legenda:
                    coluna_legenda.append(c) # Adiciona à lista as letras únicas que representam as variáveis proposicionais

            # negações explícitas
            for i, c in enumerate(expressao):
                if c.isalpha():
                    if i > 0 and expressao[i - 1] == "~":
                        token = f"~{c}" # Concatena o símbolo de negação com a variável correspondente
                        if token not in coluna_legenda:
                            coluna_legenda.append(token) # Adiciona a negação da variável como uma coluna separada na lista

            # subexpressões internas
            encontrar_relacoes(expressao,coluna_legenda) # Invoca a função recursiva para encontrar e listar as subexpressões entre parênteses

            # expressão principal
            if expressao not in coluna_legenda:
                coluna_legenda.append(expressao) # Garante que a expressão lógica completa seja a última coluna da lista de cabeçalhos
            # remover duplicatas
            coluna_legenda = list(dict.fromkeys(coluna_legenda)) # Elimina possíveis repetições mantendo a ordem original de inserção

            # ==========================
            # ORDENAÇÃO DAS VARIÁVEIS
            # ==========================

            variaveis_ordenadas = sorted(
                [x for x in coluna_legenda if len(x) == 1]
            ) # Filtra e organiza em ordem alfabética apenas as variáveis de tamanho 1 (ex: A, B, C)

            outras_colunas = [
                x for x in coluna_legenda
                if len(x) != 1
                ] # Isola as negações, subexpressões e a expressão principal que possuem tamanho maior que 1

            coluna_legenda = variaveis_ordenadas + outras_colunas # Junta as listas posicionando as variáveis ordenadas sempre nas primeiras colunas

            # ============================================================
            # GERAÇÃO DA TABELA VERDADE E CLASSIFICAÇÃO AUTOMÁTICA
            # ============================================================
            variaveis, _ = numero_variaveis_linhas(coluna_legenda) # Obtém o número de variáveis únicas ignorando o total de linhas gerado

            tabela = [coluna_legenda] # Inicializa a tabela colocando a lista de cabeçalhos na primeira linha (índice 0)
            tabela = gerar_combinacoes_bool(variaveis, tabela) # Preenche as linhas iniciais com as combinações de True/False das variáveis
            tabela = gerar_bool_subexpressões(coluna_legenda, tabela) # Calcula e preenche as colunas restantes com os valores das subexpressões

            true = 0 # Inicializa o contador para mapear quantas linhas resultaram em verdadeiro na expressão final

            for linha in tabela[1:]: # Percorre cada linha de dados pulando o cabeçalho de strings
                if linha[-1]:
                    true += 1 # Incrementa o contador se o último elemento da linha (o resultado final da expressão) for True
            if true == len(tabela[1:]):
                print('A expressão lógica é uma Tautologia') # Classifica se todas as combinações resultaram em verdadeiro
            elif true == 0:
                print('A expressão lógica é uma Contradição') # Classifica se todas as combinações resultaram em falso
            else:
                print('A expressão lógica é uma Contingência') # Classifica se o resultado final mesclar valores verdadeiros e falsos
            


            df = pd.DataFrame(tabela[1:], columns=tabela[0]) # Converte a matriz booleana em um DataFrame estruturado do Pandas
            display(df) # Renderiza a tabela de forma visual e organizada no ambiente de execução (Jupyter)
            return df # Retorna o objeto DataFrame gerado para uso externo ou manipulações posteriores
        
def comparar_tabelas():
    tabela1=programa()
    tabela2=programa()
    if len(tabela1) == len(tabela2):
        if tabela1.iloc[:, -1].equals(tabela2.iloc[:, -1]):
            print('As expressões são equivalentes')
        else:
            print('Não são equivalentes')
    else:
        print('Não são equivalentes')
def escolher():
    opção=input('O que você quer fazer?(Digite 1 para gerar 1 tabela verdade, Digite 2 para comparar duas tabelas)')
    if opção == '1':
        programa() 
    elif opção == '2':
        comparar_tabelas()
    else:
        print('Selecione uma opçaõ válida')
        escolher()      
escolher()

A expressão lógica é uma Contingência


,P,Q,R,P&Q,(P&Q)->(R)
0,False,False,False,False,True
1,False,False,True,False,True
2,False,True,False,False,True
3,False,True,True,False,True
4,True,False,False,False,True
5,True,False,True,False,True
6,True,True,False,True,False
7,True,True,True,True,True
